In [1]:
# 06_data_audit — Cell 1 (self-contained)
import os, glob, pandas as pd
RAW = "/mnt/g/banglafake-detection/data/raw"
SRC = {"banfakenews2020": f"{RAW}/BanFakeNews",
       "banfakenews2":    f"{RAW}/BanFakeNews-2.0",
       "banglafake2025":  None}   # top-level Bangla_*_News_Dataset.*

def files_of(name, d):
    if d is None:
        return sorted(glob.glob(f"{RAW}/Bangla_*_News_Dataset.*"))
    return sorted(p for p in glob.glob(d + "/**/*", recursive=True)
                  if p.lower().endswith((".csv", ".xlsx", ".xls", ".json", ".jsonl", ".parquet")))

def load(p):
    e = p.lower().rsplit(".", 1)[-1]
    if e == "csv":
        try: return pd.read_csv(p)
        except UnicodeDecodeError: return pd.read_csv(p, encoding="utf-8-sig")
    if e in ("xlsx", "xls"): return pd.read_excel(p)
    if e in ("json", "jsonl"): return pd.read_json(p, lines=(e == "jsonl"))
    if e == "parquet": return pd.read_parquet(p)

for name, d in SRC.items():
    print("\n" + "#" * 12, name)
    for p in files_of(name, d):
        df = load(p)
        print("\n==", os.path.relpath(p, RAW), df.shape)
        print(df.dtypes.to_string())
        print(df.head(1).T.astype(str).apply(lambda s: s.str[:80]).to_string())
        for c in df.columns:
            if df[c].dtype == object and (df[c].nunique() <= 30 or c.lower() in ("domain", "source")):
                print("\n--", c, "| unique:", df[c].nunique())
                print(df[c].value_counts().head(10).to_string())


############ banfakenews2020

== BanFakeNews/Authentic-48K.csv (48678, 7)
articleID    int64
domain         str
date           str
category       str
headline       str
content        str
label        int64
                                                                                          0
articleID                                                                                 1
domain                                                                       jagonews24.com
date                                                                    2018-09-19 17:48:18
category                                                                          Education
headline                                   হট্টগোল করায় বাকৃবিতে দুইজন বরখাস্ত, ৬ জনকে শোকজ
content    গত ১৭ সেপ্টেম্বর বাংলাদেশ কৃষি বিশ্ববিদ্যালয়ে (বাকৃবি) উপাচার্যের কার্যালয়ে হট্ট
label                                                                                     1

== BanFakeNews/Fake-1K.csv (1299, 7)
articleID    int64